# Figure 1: study design and glycoproteomic summary -- data preparation

In [ ]:
library(tidyverse)
library(data.table)

## phenotype data

In [ ]:

rawGlycodata <- read.table("rosmap/glycoproteomics/dlpfcTissue/batch1/values/rawValues.txt",
                           check.names = FALSE)
rawGlycodata %>% head
# phenotype_preprocessing bed
unadj <- fread('ftp_fgc_xqtl/gpQTL/unadjusted/analysis_ready/phenotype_preprocessing/unadj_glyco.sorted.bed.gz')
unadj %>% head

## Restrict to the final 316 analysis-ready samples

In [ ]:
sample_map <- fread('rosmap/wgs/ampad/experimentalData/sampleSheet_1200.csv')
unadj_cov <- fread('ftp_fgc_xqtl/gpQTL/unadjusted/analysis_ready/covariate_preprocessing/unadj_glyco.sorted.covariates_unadjusted_glyco.ROSMAP_NIA_WGS.leftnorm.bcftools_qc.plink_qc.pQTL.plink_qc.prune.pca.Marchenko_PC.gz')
final_samples <- colnames(unadj_cov) %>% .[-1]
final_samples %>% length  # 316

# rawGlycodata colnames are consistently 8-digit zero-padded ProjIDs, but sample_map$ProjID
final_pjids <- sample_map %>% filter(SampleID %in% final_samples) %>% pull(ProjID) %>% as.numeric() %>% sprintf("%08d", .)
final_pjids %>% length  # 316
stopifnot(length(final_pjids) == length(final_samples), sum(duplicated(final_pjids)) == 0,
          all(final_pjids %in% colnames(rawGlycodata)))  # all 316 present in the raw file

rawGlycodata <- rawGlycodata[, final_pjids]
dim(rawGlycodata)  # 9100 glycopeptiforms x 316 final analysis-ready samples

## annotation

In [ ]:
qc_remove_gpids <- readLines('data/qc_remove_gpids.txt')
w_sole_exclude_gpids <- readLines('data/w_sole_exclude_gpids.txt')

annotation <- readRDS('rosmap/glycoproteomics/dlpfcTissue/batch1/annotation/gpAnnotation.Rds') %>%
  filter(!gpID %in% qc_remove_gpids) %>%
  # # the corrected file dropped Remove entirely
  mutate(Remove = FALSE) %>%
  mutate(GlycanTypeSummary = case_when(
    str_detect(GlycanType, "(^|;)gc(;|$)") & str_detect(GlycanType, "(^|;)N(;|$)") ~ "mixed",
    str_detect(GlycanType, "(^|;)gc(;|$)") ~ "glycation",
    str_detect(GlycanType, "(^|;)N(;|$)")  ~ "glycosylation",
    TRUE ~ GlycanTypeSummary
  ))
annotation %>% head
annotation %>% dim

annotation <- annotation %>% filter(!gpID %in% w_sole_exclude_gpids)
annotation %>% dim  # post-exclusion: (9100 - 2 qc_remove) - 54 = 9044

annotation_unadj <- annotation %>% filter(gpID %in% rownames(rawGlycodata))%>%
  mutate(
    glycopeptiform_id = paste(UniprotFirst, Sequence, GlycanComposition, GlycoSite, sep = "_")
  )
annotation_unadj %>% dim

glycation <- annotation_unadj  %>% filter(GlycanTypeSummary == 'glycation'|GlycanTypeSummary == 'mixed') # count mixed in fig 1
glycosylation <- annotation_unadj %>% filter(GlycanTypeSummary == 'glycosylation'|GlycanTypeSummary == 'mixed')

## Fig 1b

In [ ]:
gpID_glycation <- glycation %>% pull(gpID)
gpID_glycosylation <- glycosylation %>% pull(gpID)

# Prepare list
venn_list <- list(
  Glycation = gpID_glycation,
  Glycosylation = gpID_glycosylation
)
dir.create('data/')
saveRDS(venn_list, 'data/Figure_1b.rds')

library(ggvenn)

venn_list <- readRDS('data/Figure_1b.rds')
# Plot Venn with professional colors
f1b <- ggvenn(
  venn_list,
  fill_color = c("#C44E52", "#4C72B0"),  # blue and red
  fill_alpha = 0.6,
  stroke_size = 0.5,
  set_name_size = 5,
  text_size = 4
)

## Fig 1C

In [ ]:
# --- Boxplots of detection per sample
gp <- annotation

library(dplyr)
library(tidyr)
library(tibble)
library(ggplot2)


expanded_gp <- gp %>%
  mutate(GlycanTypeSummary = ifelse(GlycanTypeSummary == "mixed", "glycation", GlycanTypeSummary)) %>%
  bind_rows(
    gp %>%
      filter(GlycanTypeSummary == "mixed") %>%
      mutate(GlycanTypeSummary = "glycosylation")
  ) %>%
  select(gpID, UniprotFirst, GlycanComposition, GlycoSite, GlycanTypeSummary, BaseSequence)

det_long <- rawGlycodata
det_long[] <- !is.na(det_long)                      # logical presence/absence
det_long <- det_long %>%
  as.data.frame() %>%
  rownames_to_column("gpID") %>%
  pivot_longer(-gpID, names_to = "Sample", values_to = "Detected") %>%
  filter(Detected) %>%
  select(-Detected)

counts_per_sample <- det_long %>%
  left_join(expanded_gp, by = "gpID", relationship = "many-to-many") %>%
  filter(!is.na(GlycanTypeSummary)) %>%
  group_by(Sample, GlycanTypeSummary) %>%
  summarise(
    Glycoproteins   = n_distinct(UniprotFirst),
    Glycopeptiforms = n_distinct(gpID),
    Glycans         = n_distinct(unlist(str_split(GlycanComposition, ";"))),
    Glycanpeptides         = n_distinct(BaseSequence),
    Glycosites      = n_distinct(unlist(str_split(GlycoSite[!is.na(GlycoSite)], ";"))),
    .groups = "drop_last"
  ) %>%
  ungroup() %>%
  pivot_longer(
    cols = c(Glycoproteins, Glycopeptiforms, Glycans, Glycanpeptides, Glycosites),
    names_to = "Feature", values_to = "Count"
  ) %>%
  mutate(
    Feature = factor(Feature, levels = c("Glycans", "Glycoproteins", "Glycosites", 'Glycanpeptides', "Glycopeptiforms")),
    GlycanTypeSummary = factor(GlycanTypeSummary, levels = c("glycation", "glycosylation"))
  )

counts_per_sample <- counts_per_sample %>%
  dplyr::mutate(
    Feature = factor(Feature, levels = c("Glycans", "Glycoproteins", "Glycosites",'Glycanpeptides',"Glycopeptiforms"))
  )

counts_per_sample <- counts_per_sample %>% filter(!(Feature == 'Glycans' & GlycanTypeSummary == 'glycation'))
counts_per_sample = counts_per_sample %>% mutate(GlycanTypeSummary = gsub('gly', 'Gly', GlycanTypeSummary))

saveRDS(counts_per_sample, 'data/Figure_1c.rds')


## Fig 1D

In [ ]:
# NOTE (see chat discussion)


glycan_class_summary <- annotation_unadj %>%
  filter(!Remove) %>%
  separate_rows(GlycanClass, sep = ";") %>%
  mutate(GlycanClass = trimws(GlycanClass)) %>%
  filter(GlycanClass %in% c("High-mannose", "Paucimannose", "Hybrid/Complex", "Other")) %>%
  group_by(GlycanClass) %>%
  summarise(n = n_distinct(gpID), .groups = "drop") %>%
  mutate(
    Percent = n / sum(n) * 100,
    Label = paste0(round(Percent, 1), "%")
  )
   saveRDS(glycan_class_summary, 'data/Figure_1d.rds')